# Tarea para el Hogar 04

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Oct 12 19:38:22 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671199,35.9,1479635,79.1,1479635,79.1
Vcells,1242607,9.5,8388608,64.0,1978693,15.1


### 2.2.2 Carga de Librerias

In [3]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: parallel

Loading required package: primes

Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm



### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [4]:

PARAM <- list()

PARAM$experimento <- 5944
PARAM$semilla_primigenia <- 149911



In [5]:
PARAM$kaggle$competencia <- "labo-1-virtual-inicial"
PARAM$kaggle$cortes <- seq(9000, 12000, by= 500)

In [6]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [7]:


# Parametros LightGBM - Experimento 5944

PARAM$hyperparametertuning$xval_folds <- 5

PARAM$lgbm$param_fijos <- list(
  boosting = "gbdt",
  objective = "binary",
  metric = "auc",
  first_metric_only = FALSE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  force_row_wise = TRUE,
  verbosity = -100,

  seed = PARAM$semilla_primigenia,

  max_depth = -1L,
  min_gain_to_split = 0,
  min_sum_hessian_in_leaf = 0.001,
  lambda_l1 = 0.0,
  lambda_l2 = 0.0,
  max_bin = 31L,

  bagging_fraction = 1.0,
  bagging_freq = 1L,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  is_unbalance = FALSE,
  scale_pos_weight = 1.0,

  drop_rate = 0.1,
  max_drop = 50,
  skip_drop = 0.5,

  extra_trees = FALSE,

  # Base del experimento 5942
  num_iterations = 1000L,
  learning_rate = 0.01,
  feature_fraction = 0.75,
  num_leaves = 64L,
  min_data_in_leaf = 500L
)


### 2.2.4  Preprocesamiento

In [8]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [9]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [10]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [11]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [12]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [13]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [14]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 83665

[1] 154

2.2.5 Configuracion del Grid Search

In [15]:
# En el argumento x llegan los parmaetros de LightGBM
#  devuelve la AUC en cross validation del modelo entrenado

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelocv <- lgb.cv(
    data= dtrain,
    nfold= PARAM$hyperparametertuning$xval_folds,
    stratified= TRUE,
    param= param_completo
  )

  # obtengo la ganancia
  AUC <- modelocv$best_score

  # hago espacio en la memoria
  rm(modelocv)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " AUC ", AUC
  )

  return(AUC)
}

In [16]:

# Grid Search - Experimento 5944

tb_nueva <- CJ(
  learning_rate = c(0.01, 0.015, 0.02, 0.03),
  num_leaves = c(32L, 48L, 64L),
  min_data_in_leaf = c(250L, 500L, 1000L),
  feature_fraction = c(0.60, 0.75),
  lambda_l1 = c(0, 0.5, 1),
  bagging_fraction = c(0.85, 1.00)
)

cat("Combinaciones:", nrow(tb_nueva), "\n")

print(tb_nueva)




Combinaciones: 432 
Key: <learning_rate, num_leaves, min_data_in_leaf, feature_fraction, lambda_l1, bagging_fraction>
     learning_rate num_leaves min_data_in_leaf feature_fraction lambda_l1
             <num>      <int>            <int>            <num>     <num>
  1:          0.01         32              250             0.60       0.0
  2:          0.01         32              250             0.60       0.0
  3:          0.01         32              250             0.60       0.5
  4:          0.01         32              250             0.60       0.5
  5:          0.01         32              250             0.60       1.0
 ---                                                                     
428:          0.03         64             1000             0.75       0.0
429:          0.03         64             1000             0.75       0.5
430:          0.03         64             1000             0.75       0.5
431:          0.03         64             1000             0.75     

In [17]:
# veo que tiene la tabla antes de procesar
tb_nueva

learning_rate,num_leaves,min_data_in_leaf,feature_fraction,lambda_l1,bagging_fraction
<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>
0.01,32,250,0.60,0.0,0.85
0.01,32,250,0.60,0.0,1.00
0.01,32,250,0.60,0.5,0.85
0.01,32,250,0.60,0.5,1.00
0.01,32,250,0.60,1.0,0.85
0.01,32,250,0.60,1.0,1.00
0.01,32,250,0.75,0.0,0.85
0.01,32,250,0.75,0.0,1.00
0.01,32,250,0.75,0.5,0.85


In [18]:
# registro a registro calculo la AUC

tb_nueva[, AUC := Estimar_AUC_lightgbm(as.list(.SD)),
  by = 1:nrow(tb_nueva)]

Mon Oct 12 19:39:02 2026  0.01, 32, 250, 0.6, 0, 0.85 AUC 0.932362498836477

Mon Oct 12 19:39:35 2026  0.01, 32, 250, 0.6, 0, 1 AUC 0.931393607457581

Mon Oct 12 19:40:12 2026  0.01, 32, 250, 0.6, 0.5, 0.85 AUC 0.932340979817991

Mon Oct 12 19:40:48 2026  0.01, 32, 250, 0.6, 0.5, 1 AUC 0.931752761423622

Mon Oct 12 19:41:26 2026  0.01, 32, 250, 0.6, 1, 0.85 AUC 0.932377012082184

Mon Oct 12 19:42:01 2026  0.01, 32, 250, 0.6, 1, 1 AUC 0.933001005067626

Mon Oct 12 19:42:33 2026  0.01, 32, 250, 0.75, 0, 0.85 AUC 0.932959708852713

Mon Oct 12 19:43:04 2026  0.01, 32, 250, 0.75, 0, 1 AUC 0.930407311742744

Mon Oct 12 19:43:37 2026  0.01, 32, 250, 0.75, 0.5, 0.85 AUC 0.933151574118111

Mon Oct 12 19:44:06 2026  0.01, 32, 250, 0.75, 0.5, 1 AUC 0.931649374923479

Mon Oct 12 19:44:39 2026  0.01, 32, 250, 0.75, 1, 0.85 AUC 0.93342756809475

Mon Oct 12 19:45:12 2026  0.01, 32, 250, 0.75, 1, 1 AUC 0.931700041425153

Mon Oct 12 19:45:46 2026  0.01, 32, 500, 0.6, 0, 0.85 AUC 0.932161509754973

Mon 

In [19]:
# veo que tiene la tabla DESPUES de procesar
tb_nueva

learning_rate,num_leaves,min_data_in_leaf,feature_fraction,lambda_l1,bagging_fraction,AUC
<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
0.01,32,250,0.60,0.0,0.85,0.9323625
0.01,32,250,0.60,0.0,1.00,0.9313936
0.01,32,250,0.60,0.5,0.85,0.9323410
0.01,32,250,0.60,0.5,1.00,0.9317528
0.01,32,250,0.60,1.0,0.85,0.9323770
0.01,32,250,0.60,1.0,1.00,0.9330010
0.01,32,250,0.75,0.0,0.85,0.9329597
0.01,32,250,0.75,0.0,1.00,0.9304073
0.01,32,250,0.75,0.5,0.85,0.9331516


In [20]:
fwrite( tb_nueva,
  file= "tb_grid_search_05.txt",
  sep="\t",
  append= TRUE
)

In [21]:
setorder( tb_nueva, -AUC)
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC]
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$learning_rate
[1] 0.01

$num_leaves
[1] 48

$min_data_in_leaf
[1] 250

$feature_fraction
[1] 0.6

$lambda_l1
[1] 1

$bagging_fraction
[1] 0.85

In [22]:
write_yaml( PARAM, file="PARAM.yml")

In [23]:


# Analisis de importancia - Experimento 5944

hiperparametros <- c(
  "learning_rate",
  "num_leaves",
  "min_data_in_leaf",
  "feature_fraction",
  "lambda_l1",
  "bagging_fraction"
)

importancia <- rbindlist(lapply(hiperparametros, function(h) {

  resumen <- tb_nueva[
    , .(AUC_media = mean(AUC)),
    by = h
  ]

  data.table(
    hiperparametro = h,
    impacto_AUC = max(resumen$AUC_media) -
                  min(resumen$AUC_media)
  )
}))

setorder(importancia, -impacto_AUC)

cat("\nIMPORTANCIA DE HIPERPARAMETROS\n")
print(importancia)

cat("\nMEJORES 10 COMBINACIONES\n")
print(head(tb_nueva, 10))

fwrite(importancia, "importancia_5944.csv")




IMPORTANCIA DE HIPERPARAMETROS
     hiperparametro  impacto_AUC
             <char>        <num>
1: min_data_in_leaf 0.0006756839
2:    learning_rate 0.0004500806
3: feature_fraction 0.0003473368
4: bagging_fraction 0.0002647448
5:        lambda_l1 0.0002547005
6:       num_leaves 0.0002294129

MEJORES 10 COMBINACIONES
    learning_rate num_leaves min_data_in_leaf feature_fraction lambda_l1
            <num>      <int>            <int>            <num>     <num>
 1:         0.010         48              250             0.60       1.0
 2:         0.015         48              250             0.60       1.0
 3:         0.015         32              250             0.60       0.0
 4:         0.010         32              250             0.75       1.0
 5:         0.020         64              500             0.60       0.0
 6:         0.015         32              500             0.75       0.0
 7:         0.010         32             1000             0.60       1.0
 8:         0.010    

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [24]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparametros

In [25]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [26]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [27]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [28]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 149911

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 1

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 0.85

$bagging_freq
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 1000

$learning_rate
[1] 0.01

$feature_fraction
[1] 0.6

$num_leaves
[1] 48

$min_data_in_leaf
[1] 250

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [29]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [30]:
  # entreno LightGBM

  modelo_final <- lgb.train(
    data= dtrain,
    param= param_normalizado
  )

In [31]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(modelo_final))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

In [32]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [33]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [34]:
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

Kaggle Competition Submit

In [35]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
  Sys.sleep(45)
}

99 submissions remaining today. Successfully submitted to Labo 1 Virtual Inicial 
98 submissions remaining today. Successfully submitted to Labo 1 Virtual Inicial 
97 submissions remaining today. Successfully submitted to Labo 1 Virtual Inicial 
96 submissions remaining today. Successfully submitted to Labo 1 Virtual Inicial 
95 submissions remaining today. Successfully submitted to Labo 1 Virtual Inicial 
94 submissions remaining today. Successfully submitted to Labo 1 Virtual Inicial 
93 submissions remaining today. Successfully submitted to Labo 1 Virtual Inicial 


In [36]:
write_yaml( PARAM, file="PARAM.yml")

In [37]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Oct 13 00:37:12 2026"

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-04**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-04**

Siéntase libre de agregar columnas a la hoja **TareaHogar-04**  en caso de ser necesario.